In [ ]:
import pandas as pd

# load corpus

df = pd.read_csv(
    "polityka_articles.csv",
    low_memory=False
)

old_df = pd.read_csv(
    "polityka_ukraine_russia.csv",
    low_memory=False
)

print("=" * 80)

print("rows")

print(len(df))

print()

print("columns")

for c in df.columns:

    print(c)

print()

print("=" * 80)

# check missing values

summary = []

for col in df.columns:

    missing = df[col].isna().sum()

    empty = 0

    if df[col].dtype == object:

        empty = (

            df[col]
            .fillna("")
            .str.strip()
            .eq("")
            .sum()

        )

    summary.append({

        "column": col,
        "missing": missing,
        "empty_strings": empty,
        "filled": len(df) - max(missing, empty)

    })

summary = pd.DataFrame(summary)

print(summary.sort_values("missing", ascending=False))

print()

print("=" * 80)

# check important fields

important = [

    "url",
    "title",
    "publish_date",
    "text"

]

print()

print("important fields")

print()

for field in important:

    if field in df.columns:

        print(f"{field:20} ok")

    else:

        print(f"{field:20} missing")

In [ ]:
# filter articles about russia

russia_keywords = [

    # Russia

    r"\brosj\w*\b",

    # Kremlin

    r"\bkreml\w*\b",

    # Moscow

    r"\bmoskw\w*\b",

    # Putin

    r"\bputin\b",

    # Russian officials

    r"\bławrow\w*\b",
    r"\blawrow\w*\b",

    r"\bpieskow\w*\b",

    r"\bszojgu\w*\b",

    r"\bgierasimow\w*\b",
    r"\bgerasimow\w*\b",

    r"\bmiedwiediew\w*\b",

    r"\bzacharow\w*\b",

    r"\bmiszustin\w*\b",

    # Companies

    r"\bgazprom\b",

    r"\brosnieft\b",

    r"\brosatom\b",

    r"\btransnieft\b",

    r"\błukoil\b",
    r"\blukoil\b",

    # Institutions

    r"\bduma\b",

    # Crimea

    r"\bkrym\b",

    # War names

    r"\bwojna (na|w) ukrainie\b",

    r"\bwojna rosji z ukrainą\b",
    r"\bwojna rosji z ukraina\b",

    r"\brosyjska agresja\b",
    r"\bagresja rosji\b"

]

pattern = "|".join(russia_keywords)

second_df = df[

    (

        df["title"]
        .fillna("")
        .str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )

    )

    |

    (

        df["text"]
        .fillna("")
        .str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )

    )

].copy()

print(len(second_df))

In [ ]:
# count articles matched by each keyword

keyword_counts = {}

text = (

    df["title"]
    .fillna("")
    + " "
    + df["text"]
    .fillna("")

)

for keyword in russia_keywords:

    keyword_counts[keyword] = (

        text
        .str.contains(
            keyword,
            case=False,
            na=False,
            regex=True
        )
        .sum()

    )

keyword_counts = (

    pd.Series(keyword_counts)
    .sort_values(ascending=False)
    .reset_index()

)

keyword_counts.columns = [

    "keyword",
    "articles"

]

display(keyword_counts)

In [ ]:
# compare filtered corpus with old corpus

old_urls = set(
    old_df["url"]
)

new_urls = set(
    second_df["url"]
)

print(f"old corpus: {len(old_urls):,}")

print(f"new corpus: {len(new_urls):,}")

print()

print(f"overlap: {len(old_urls & new_urls):,}")

print(f"only old: {len(old_urls - new_urls):,}")

print(f"only new: {len(new_urls - old_urls):,}")

In [ ]:
# save second round corpus

second_df.to_csv(

    "polityka_SECONDROUD_articles.csv",

    index=False,

    encoding="utf-8-sig"

)

In [ ]:
import os
import shutil
import pandas as pd
from tqdm import tqdm

# load data

articles = pd.read_csv(
    "polityka_ukraine_russia_with_images.csv",
    low_memory=False
)

index = pd.read_csv(
    "polityka_images_index.csv",
    low_memory=False
)

os.makedirs(
    "polityka_SECONDROUND_images",
    exist_ok=True
)

overlap_urls = set(
    old_df["url"]
) & set(
    second_df["url"]
)

print(len(overlap_urls))

# keep overlap articles

overlap_df = articles[

    articles["url"].isin(
        overlap_urls
    )

].copy()

print(len(overlap_df))


# keep overlap articles

overlap_df = articles[

    articles["url"].isin(
        overlap_df["url"]
    )

].copy()

print(len(overlap_df))

# collect unique image urls

image_urls = set()

for urls in overlap_df["image_url"].fillna(""):

    for url in str(urls).split("|"):

        url = url.strip()

        if url:

            image_urls.add(url)

print()

print("unique images")

print(len(image_urls))

# keep only images that exist in the index

index = index[

    index["image_url"].isin(
        image_urls
    )

].copy()

print()

print("images found")

print(len(index))

# copy images

source_folder = (
    "."
)

output_folder = (
    "polityka_SECONDROUND_images"
)

copied = 0

missing = []

for _, row in tqdm(
    index.iterrows(),
    total=len(index)
):

    source = os.path.join(
        source_folder,
        row["filename"]
    )

    destination = os.path.join(
        output_folder,
        row["filename"]
    )

    if not os.path.exists(source):

        missing.append(row)

        continue

    shutil.copy2(
        source,
        destination
    )

    copied += 1

print()

print(f"copied: {copied:,}")

print(f"missing: {len(missing):,}")

In [ ]:
image_index = pd.read_csv(
    "polityka_images_index.csv",
    encoding="utf-8-sig"
)

# create mapping

image_mapping = dict(

    zip(

        image_index["image_url"],
        image_index["filename"]

    )

)

# create image filenames

image_filenames = []

for urls in overlap_df["image_url"].fillna(""):

    filenames = []

    for url in str(urls).split("|"):

        url = url.strip()

        if not url:
            continue

        if url in image_mapping:

            filenames.append(
                image_mapping[url]
            )

    image_filenames.append(

        "|".join(filenames)

    )

overlap_df["image_filename"] = image_filenames

# save final corpus

overlap_df.to_csv(

    "polityka_SECONDROUND_final_articles.csv",

    index=False,

    encoding="utf-8-sig"

)

print("saved")

print(len(overlap_df))

print()

print(

    overlap_df[[

        "url",
        "image_url",
        "image_filename"

    ]].head()

)

In [ ]:
overlap_df.head()

In [ ]:
import re
import pandas as pd

# load corpus

df = pd.read_csv(
    "polityka_SECONDROUND_final_articles.csv",
    low_memory=False
)

# load image index

image_index = pd.read_csv(
    "polityka_images_index.csv",
    low_memory=False
)

# keep only largest image

image_index["base_url"] = (

    image_index["image_url"]
    .str.replace(
        r"_f\d+x\d+$",
        "",
        regex=True
    )

)

image_index = (

    image_index
    .sort_values("image_url")
    .drop_duplicates(
        subset="base_url",
        keep="first"
    )

)

print("remaining images")

print(len(image_index))

# create mapping

image_mapping = dict(

    zip(

        image_index["base_url"],
        image_index["filename"]

    )

)

# update image urls

new_image_urls = []

new_image_filenames = []

for urls in df["image_url"].fillna(""):

    article_urls = []

    article_filenames = []

    seen = set()

    for url in str(urls).split("|"):

        url = url.strip()

        if not url:
            continue

        base_url = re.sub(
            r"_f\d+x\d+$",
            "",
            url
        )

        if base_url in seen:
            continue

        seen.add(base_url)

        article_urls.append(
            base_url
        )

        if base_url in image_mapping:

            article_filenames.append(
                image_mapping[base_url]
            )

    new_image_urls.append(
        "|".join(article_urls)
    )

    new_image_filenames.append(
        "|".join(article_filenames)
    )

df["image_url"] = new_image_urls

df["image_filename"] = new_image_filenames

In [ ]:
df.to_csv(

    "polityka_SECONDROUND_final_articles.csv",

    index=False,

    encoding="utf-8-sig"

)

print()

print("saved")

print(df[[
    "image_url",
    "image_filename"
]].head())

In [ ]:
# folder

image_folder = (
    "."
)

# collect filenames that should remain

needed_files = set()

for filenames in df["image_filename"].fillna(""):

    for filename in str(filenames).split("|"):

        filename = filename.strip()

        if filename:

            needed_files.add(filename)

print("needed")

print(len(needed_files))

In [ ]:
# remove unused images

removed = 0

for filename in tqdm(
    os.listdir(image_folder)
):

    if filename not in needed_files:

        os.remove(
            os.path.join(
                image_folder,
                filename
            )
        )

        removed += 1

print()

print("removed")

print(removed)

print()

print("remaining")

print(len(os.listdir(image_folder)))

In [ ]:
import os
import pandas as pd

# load image index

image_index = pd.read_csv(
    "polityka_images_index.csv",
    low_memory=False
)

# keep only files that still exist

image_folder = (
    "polityka_SECONDROUND_images"
)

existing_files = set(
    os.listdir(image_folder)
)

image_index = image_index[
    image_index["filename"].isin(
        existing_files
    )
].copy()

# create mapping

image_mapping = dict(

    zip(

        image_index["image_url"],
        image_index["filename"]

    )

)

# create image filenames

image_filenames = []

for urls in overlap_df["image_url"].fillna(""):

    filenames = []

    for url in str(urls).split("|"):

        url = url.strip()

        if url in image_mapping:

            filenames.append(
                image_mapping[url]
            )

    image_filenames.append(
        "|".join(filenames)
    )

# create final dataframe

final_df = overlap_df.copy()

final_df["image_filename"] = image_filenames

# save final corpus

final_df.to_csv(

    "polityka_SECONDROUND_final_articles.csv",

    index=False,

    encoding="utf-8-sig"

)

print(final_df.shape)

print(final_df.columns)

In [ ]:
final_df = ва 

In [ ]:
import pandas as pd

# load image index

image_index = pd.read_csv(
    "polityka_images_index.csv",
    low_memory=False
)

# keep only files that exist in the final folder

existing_files = set(

    df["image_filename"]
    .fillna("")
    .str.split("|")
    .explode()
    .str.strip()

)

existing_files.discard("")

image_index = image_index[

    image_index["filename"].isin(
        existing_files
    )

].copy()

# create filename -> url mapping

filename_mapping = dict(

    zip(

        image_index["filename"],
        image_index["image_url"]

    )

)

# rebuild image_url

new_image_urls = []

for filenames in df["image_filename"].fillna(""):

    urls = []

    for filename in str(filenames).split("|"):

        filename = filename.strip()

        if filename in filename_mapping:

            urls.append(
                filename_mapping[filename]
            )

    new_image_urls.append(
        "|".join(urls)
    )

df["image_url"] = new_image_urls

print()

print("unique images")

unique_images = set()

for urls in df["image_url"].fillna(""):

    for url in str(urls).split("|"):

        url = url.strip()

        if url:

            unique_images.add(url)

print(len(unique_images))

# save final corpus

df.to_csv(

    "polityka_SECONDROUND_final_articles.csv",

    index=False,

    encoding="utf-8-sig"

)